In [10]:
import kagglehub

# Esto descarga TODO el dataset (imágenes y CSVs) y te dice en qué carpeta de tu PC se ha guardado
path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")

print("El dataset se ha descargado en:", path)

100%|██████████| 5.20G/5.20G [08:06<00:00, 11.5MB/s]

Extracting files...


El dataset se ha descargado en: C:\Users\CARLOS\.cache\kagglehub\datasets\kmader\skin-cancer-mnist-ham10000\versions\2


In [16]:
#Imports
import os
import pandas as pd

# Skin Wounds Segmentation with PyTorch

### By Carlos Parra Camacho

----

## 1. Introduction 

For this project, I used Deep Learning techniques to segment skin lesions from the Kaggle HAM10000 dataset. This dataset provides 10,000 dermatoscopic images from various populations, acquired and stored through different modalities.

The goal is to isolate the lesion from the background to provide better visibility, helping to determine whether the wound is improving or if it requires medical attention.

---

# 2. Data

In [18]:
# 1. We define the directory where the data is
data_dir = r'C:\Users\CARLOS\WoundSegmentation\data'

# 2. Join the folder to the csv file
csv_path = os.path.join(data_dir, 'HAM10000_metadata.csv')

# 3. load the file with Pandas using the hole path
dataset = pd.read_csv(csv_path)

# 4. Show the head
print(dataset.head())

     lesion_id      image_id   dx dx_type   age   sex localization
0  HAM_0000118  ISIC_0027419  bkl   histo  80.0  male        scalp
1  HAM_0000118  ISIC_0025030  bkl   histo  80.0  male        scalp
2  HAM_0002730  ISIC_0026769  bkl   histo  80.0  male        scalp
3  HAM_0002730  ISIC_0025661  bkl   histo  80.0  male        scalp
4  HAM_0001466  ISIC_0031633  bkl   histo  75.0  male          ear


In [19]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10015 entries, 0 to 10014
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   lesion_id     10015 non-null  object 
 1   image_id      10015 non-null  object 
 2   dx            10015 non-null  object 
 3   dx_type       10015 non-null  object 
 4   age           9958 non-null   float64
 5   sex           10015 non-null  object 
 6   localization  10015 non-null  object 
dtypes: float64(1), object(6)
memory usage: 547.8+ KB


## 2.1 Summary of Variables within the data

The HAM10000 dataset has 7 columns and 10015 rows in total. All variables are explained in this section, their units, their types and some descriptive statistics.

* Lesion_id - ID
    * Number of missing values = 0

* Image_id - Image Identifier
    * Number of missing values = 0

* dx - Diagnose
    * Number of missing values = 0
    * Contains 7 different values:
        - nv = Neveus melanocítico (Benign, common everydaty moles)
        - mel = Melanoma (Malignant, most dangerous skin cancer)
        - bcc = Basal cell carcinoma (Malignant/Invasive, but slow growing)
        - akiec = Actinic keratoses and intraepithelial carcinoma / Boewn's disease (Pre-cancerous)
        - bkl = Benign keratosis-like lesions (Seborrheic keratoses, non-cancerous warts)
        - df = Dermatofibroma (Benign, a type of skin nodule)
        - vasc = Vascular lesions (Benign, such as angiomas or pyogenic granulomas)
    
* dx_type - Confirmation Method
    * Number of missing values = 0
    * Contains 4 different values_
        - histopathology = Biopsia (The most reliable method; a tissue sample was cut and analyzed in a lab)
        - confocal = In-vivo confocal microscopy (Advanced live optical examination).
        - consensus = Expert consensus (Multiple dermatologists agreed on thhe diagnosis by looking at the image)
        - follow_ip = Monitoring (The lesion was monitored over months/years and did not change, confirming it was benign)

* age - Age
    * Number of missing values = 57
    * Contains patients between 0 and 85 years old

* sex - Sex
    * Number of missing values = 0
    * Contains 3 different values
        - Male
        - Female
        - Unknown

* localization - Location on the body
    * Number of missing values = 0
    * Contains 15 different values
        - back
        - lower extremity
        - trunk
        - upper extremity
        - abdomen
        - face
        - chest
        - foot
        - unknown
        - neck
        - scalp
        - hand
        - ear
        - genital
        - acral

## 2.3 dx class unbalance

As we can see, the dx which would be the target variable for a classification algorithm is pretty unbalanced which would be a problem. The big mayority of the patients had Benign moles.

In [26]:
print(dataset['dx'].value_counts())

dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64


## 2.4 Duplicated id values

# 3. Data Pre-Processing 

first of all we give the dataframe columns a conventional name (lowercase, whole words, etc)

In [29]:
dataset = dataset.rename(columns={
    'lesion_id' : 'id',
    'image_id' : 'image_id',
    'dx' : 'diagnose',
    'dx_type' : 'diagnose_method'
})

In [30]:
dataset.head(5)

,id,image_id,diagnose,diagnose_method,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [ ]:
# One-hot encode sex column